# picomake - mlp

In [9]:
import torch
import torch.nn.functional as F 
import matplotlib.pyplot as plt
%matplotlib inline

In [10]:
words = open('names.txt', 'r').read().split()
len(words)

32033

In [11]:
# build vocab

chars = set(sorted(letter for w in words for letter in w))

stoi = {s:i+1 for i,s in enumerate(sorted(chars))}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [59]:
# build out dataset

context_size = 3
X = []
Y = []

for w in words[:1]:
    print(w)
    context = [0]*context_size

    for i in w+'.':
        ix = stoi[i]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context [1:] + [ix]

X = torch.tensor(X) # X stores the context of one training example in chunk of context size (stoi)
Y = torch.tensor(Y) # Y stores the stoi value of each char in the word


emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .


In [60]:
print(X.shape)
print(Y.shape)


torch.Size([5, 3])
torch.Size([5])


In [61]:
# let us create an embedding matrix where every character gets a 2 dimensional embedding using the awesome pytorch indexing

C = torch.randn((27,2))
emb = C[X] # see below for the result
C

tensor([[-1.3970,  0.7424],
        [-1.1660,  0.7059],
        [ 0.0994,  0.3708],
        [-0.7455, -0.2672],
        [ 0.5274, -0.7583],
        [-0.6020,  1.4734],
        [ 0.0403,  0.6447],
        [-1.8046, -0.8722],
        [ 0.0077, -0.3818],
        [-0.3179,  0.8145],
        [ 0.5791,  0.0795],
        [ 0.8160, -2.1677],
        [-0.3481,  0.2965],
        [ 0.4923, -1.0870],
        [-0.9754,  0.3812],
        [-0.7559, -2.4122],
        [-0.2498, -1.3144],
        [-1.7172, -1.9362],
        [-0.1138,  0.0399],
        [ 1.0776, -1.0381],
        [-0.7994, -0.6886],
        [-0.0888,  0.7516],
        [-0.6412,  1.2904],
        [ 3.2701, -0.1560],
        [-0.1111, -0.8736],
        [ 0.8978, -0.2859],
        [ 0.1948,  0.2400]])

In [72]:
''' 
the shape is [5,3,2] 
as C[X] uses X = tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1]]) as indices into C
so we get the 0th index of C  3 times as [0,0,0] and we do this 5 times as 
length of X is 5 in the second dim

'''
print(emb.shape)
print(X)
emb

torch.Size([5, 3, 2])
tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1]])


tensor([[[-1.3970,  0.7424],
         [-1.3970,  0.7424],
         [-1.3970,  0.7424]],

        [[-1.3970,  0.7424],
         [-1.3970,  0.7424],
         [-0.6020,  1.4734]],

        [[-1.3970,  0.7424],
         [-0.6020,  1.4734],
         [ 0.4923, -1.0870]],

        [[-0.6020,  1.4734],
         [ 0.4923, -1.0870],
         [ 0.4923, -1.0870]],

        [[ 0.4923, -1.0870],
         [ 0.4923, -1.0870],
         [-1.1660,  0.7059]]])

In [75]:
W1 = torch.randn((6, 100)) # 6,100 wants to get multiplied to torch.Size([5, 3, 2]) first word say
b1 = torch.randn(100)


# we can do torch.cat([emb[:,0,:], emb[:,1,:], emb[:,2,:]], 1) or

torch.cat(torch.unbind(emb, 1), 1).shape


torch.Size([5, 6])